# NB_CursorUsage_Bronze_To_Silver
**Source:** `AI_Medallion.Bronze.Cursor_CursorUsage`  
**Targets (reads):** `OktaUserforAI`, `HC_Silver_Historical`  
**Target:** `AI_Medallion.Silver.CursorUsage`  
**Run after:** NB_OktaUserforAI_Bronze_To_Silver, NB_HCHistorical_Bronze_To_Silver

In [8]:
%run common_bootstrap

StatementMeta(, bb9bb06d-c4e0-40aa-b603-8e642e55f61c, 22, Finished, Available, Finished, True)

In [2]:
%%sql
select distinct filename from AI_Medallion.Bronze.cursor_usage

StatementMeta(, bb9bb06d-c4e0-40aa-b603-8e642e55f61c, 8, Finished, Available, Finished, False)

<Spark SQL result set with 8 rows and 1 fields>

In [9]:
# team-usage-events-24970505-2026-09-25.csv ---------
# team-usage-events-24970305-2026-09-25.csv
# team-usage-events-24969905-2026-09-25.csv
# team-usage-events-24969805-2026-09-25.csv
# team-usage-events-24970005-2026-09-25.csv
# team-usage-events-24970405-2026-09-25.csv
# team-usage-events-24970705-2026-09-25.csv
# team-usage-events-24970105-2026-09-25.csv
bronze = spark.table(TABLE_CURSOR_BRONZE).filter( F.col("filename") == "team-usage-events-24970505-2026-09-25.csv")

okta = spark.table(TABLE_OKTA_SILVER)
hc = spark.table(TABLE_HC_SILVER)

user_norm = F.lower(F.trim(F.col("User")))

StatementMeta(, bb9bb06d-c4e0-40aa-b603-8e642e55f61c, 23, Finished, Available, Finished, False)

In [10]:
base = (
    bronze.filter(is_valid_email(F.col("User")))
    .select(
        F.col("Date").cast("date").alias("CursorUsageDate"),
        user_norm.alias("CursorUsageUser"),
        F.col("`Service Account Name`").alias("CursorUsageServiceAccountName"),
        F.col("`Service Account ID`").alias("CursorUsageServiceAccountID"),
        F.col("`Cloud Agent ID`").alias("CursorUsageCloudAgentID"),
        F.col("`Automation ID`").alias("CursorUsageAutomationID"),
        F.col("Kind").alias("CursorUsageKind"),
        F.col("Model").alias("CursorUsageModel"),
        F.col("`Max Mode`").alias("CursorUsageMaxMode"),
        F.col("`Input (w/ Cache Write)`").cast("long").alias("CursorUsageInputWithCacheWrite"),
        F.col("`Input (w/o Cache Write)`").cast("long").alias("CursorUsageInputWithoutCacheWrite"),
        F.col("`Cache Read`").cast("long").alias("CursorUsageCacheRead"),
        F.col("`Output Tokens`").cast("long").alias("CursorUsageOutputTokens"),
        F.col("`Total Tokens`").cast("long").alias("CursorUsageTotalTokens"),
        F.col("Cost").cast("decimal(18,8)").alias("CursorUsageCost"),
        F.col("FileName").alias("CursorUsageFileName"),
        F.col("ExecutionDate").cast("timestamp").alias("CursorUsageIngestionDate"),
    )
    .withColumn(
        "CursorUsageMonthNo",
        (F.year("CursorUsageDate") * F.lit(100) + F.month("CursorUsageDate")).cast("int"),
    )
)

okta_slim = okta.select(
    "OktaUserId",
    "OktaUserEmail",
    "OktaUserFullEmail",
    "OktaUserEmployeeNumber",
    "OktaUserFullName",
    "OktaUserRegion",
    "OktaUserCountry",
)

# Inner join: match login or full email
joined = base.join(
    okta_slim,
    (F.col("CursorUsageUser") == F.col("OktaUserEmail"))
    | (F.col("CursorUsageUser") == F.col("OktaUserFullEmail")),
    "inner",
)

joined = joined.withColumn(
    "CursorUsageKeyEmail",
    F.when(
        F.col("OktaUserEmail").isNotNull(),
        F.concat(
            F.col("CursorUsageMonthNo").cast("string"),
            F.lit("/"),
            F.col("OktaUserEmail"),
        ),
    ),
).withColumn(
    "CursorUsageKeyFullEmail",
    F.when(
        F.col("OktaUserFullEmail").isNotNull(),
        F.concat(
            F.col("CursorUsageMonthNo").cast("string"),
            F.lit("/"),
            F.col("OktaUserFullEmail"),
        ),
    ),
)

hc_slim = hc.select(
    F.col("HCKeyEmail"),
    F.col("HCBusinessUnit"),
    F.col("HCBusinessUnitCode"),
    F.col("HCDivision"),
    F.col("HCJobTitle"),
    F.col("HCOffice"),
    F.col("HCTalentSegment"),
    F.col("Month_No").alias("HCMonthNo"),
    F.col("OktaUserCountry").alias("HCOktaUserCountry"),
    F.col("HCDivisionINC"),
)

joined = joined.join(
    hc_slim,
    (F.col("HCKeyEmail") == F.col("CursorUsageKeyEmail"))
    | (F.col("HCKeyEmail") == F.col("CursorUsageKeyFullEmail")),
    "left",
).withColumn(
    "HCDivisionINC",
    F.coalesce(F.col("HCDivisionINC"), F.lit(HC_DIVISION_INC_DEFAULT)),
)

write_df= joined.distinct()

write_df=write_df.withColumn(
    FACT_CURSOR_ACTIVE_MERGE_KEY,
    cursor_usage_record_key()
    )

StatementMeta(, bb9bb06d-c4e0-40aa-b603-8e642e55f61c, 24, Finished, Available, Finished, False)

In [11]:
merge_incremental(
    spark,
    write_df,
    TABLE_CURSOR_SILVER,
    FACT_CURSOR_ACTIVE_MERGE_KEY,
)

StatementMeta(, bb9bb06d-c4e0-40aa-b603-8e642e55f61c, 25, Finished, Cancelled, Cancelled, False)

Py4JJavaError: An error occurred while calling o5704.saveAsTable.
: org.apache.spark.SparkException: Job 29 cancelled part of cancelled job group 25
	at org.apache.spark.scheduler.DAGScheduler.failJobAndIndependentStages(DAGScheduler.scala:3241)
	at org.apache.spark.scheduler.DAGScheduler.handleJobCancellation(DAGScheduler.scala:3111)
	at org.apache.spark.scheduler.DAGScheduler.$anonfun$handleJobGroupCancelled$4(DAGScheduler.scala:1278)
	at scala.runtime.java8.JFunction1$mcVI$sp.apply(JFunction1$mcVI$sp.java:23)
	at scala.collection.mutable.HashSet.foreach(HashSet.scala:79)
	at org.apache.spark.scheduler.DAGScheduler.handleJobGroupCancelled(DAGScheduler.scala:1277)
	at org.apache.spark.scheduler.DAGSchedulerEventProcessLoop.doOnReceive(DAGScheduler.scala:3403)
	at org.apache.spark.scheduler.DAGSchedulerEventProcessLoop.onReceive(DAGScheduler.scala:3379)
	at org.apache.spark.scheduler.DAGSchedulerEventProcessLoop.onReceive(DAGScheduler.scala:3368)
	at org.apache.spark.util.EventLoop$$an

In [7]:
# display(write_df.limit(10))
merge_incremental(
    spark,
    write_df,
    TABLE_CURSOR_SILVER,
    FACT_CURSOR_ACTIVE_MERGE_KEY,
)

# display(write_df.limit(10))

StatementMeta(, bb9bb06d-c4e0-40aa-b603-8e642e55f61c, 17, Finished, Available, Finished, False)

AnalysisException: [DELTA_TABLE_NOT_FOUND] Delta table `chimcobldhq2agrf8kg42tbkdsg2q8248lb2aga9bt6map31dhm6irre4l9mir3mclp0`.`cursor_usage` doesn't exist.

In [12]:
%%sql
drop table AI_Medallion.Bronze.Cursor_CursorUsage

StatementMeta(, 589b5309-e264-4acd-aa3f-d771561c52c1, 19, Finished, Available, Finished, False)

Error: [TABLE_OR_VIEW_NOT_FOUND] The table or view `spark_catalog`.`chimcobldhq2agrf8kg42tbkdsg2q8248lb2aga9bt6map31dhm6irre4l174rref9ig`.`Cursor_CursorUsage` cannot be found. Verify the spelling and correctness of the schema and catalog.
If you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.
To tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE IF EXISTS.